# Colab 35 — Final SNNEED vs ESM-2 vs Dice (deck run of record)

**This is the number-generating notebook the presentation redo is gated on.** It replaces
`colab33_regpool_vs_baselines.ipynb`, which is **void**: that run produced 3Di Spearman 0.33 and blank AA
AUROC/MAP@10 from a partial oracle build. colab34 re-ran the identical configuration and got 3Di 0.96
with a healthy AA oracle (10 queries / 5 positive pairs), and its `clf-pool` arm reproduced the deck
(colab29b) to within seed noise. **Do not cite `colab33_metrics.csv` or its figure.**

### The model this notebook deploys

`colab34` settled the objective and the loss weighting:

| Question | Answer | Evidence |
|---|---|---|
| Classifier or regression? | **Regression.** Equal or better on every feed; the only effect that survives seed noise is AA, favouring regression (per-seed clf `0.00 / -0.00 / 0.13` vs reg `0.19 / 0.17 / 0.15`, no overlap). Value fidelity is better everywhere (SS RMSE 0.060 vs 0.123). | colab34, replicating colab32 |
| Band weights `0.5 / 2.0 / 4.0` or flat? | **Flat.** All weighting deltas were within seed noise, because the far band contains **5 of 30,000 training pairs** — `w_far` has been applied to 5 examples since colab14. Nothing to weight. | colab34 training log |

So the deployed encoder is the simplest form the ablations permit:

> **SNNEED** = embedding -> 2x Conv1d -> `AdaptiveAvgPool1d(K=16)` -> Linear -> L2-normalise,
> trained with **plain MSE** on `normLev` through the parameter-free readout `1 - ||e_a - e_b|| / 2`.
> No head. No class bins. No loss weights.

Every knob that was removed has an ablation showing removal cost nothing. That is what makes this
defensible in a "nackte Fakten" Methods chapter.

---

### The three methods, and what each one is *for*

| Method | Role | Claim it supports |
|---|---|---|
| **SNNEED** (ours) | task-specific encoder, ~0.3M params, trained on synthetic AA only | the deliverable |
| **ESM-2** `esm2_t12_35M_UR50D`, frozen | **baseline on AA**, **control on SS/3Di** | see below |
| **Dice** 3-gram set overlap | classical, no learning | what a learned metric buys over a cheap classical one |

**The ESM-2 framing matters and is not negotiable after the last talk.**

- **On AA it is a baseline.** The question it answers is a decision question, not a contest:
  *"If you already have a protein LM in your pipeline, can you just use its cosine for edit-distance
  retrieval?"* Fenoy et al. (2022) report rho = 0.66 between PLM cosine and **BLASTp local identity** on a
  human CAFA3 subset that was **not** redundancy-reduced — that licenses *including* ESM-2, it does not
  license expecting 0.66 here, because our target is **global** normalized Levenshtein and our pool is
  redundancy-reduced at 20% identity.
- **On SS/3Di it is a control, not a baseline.** ESM-2's tokenizer reads `H`, `L`, `S` as histidine,
  leucine, serine. So does SNNEED — `CHAR_TO_IDX` in this notebook maps only the 20 amino-acid letters,
  and the 3Di alphabet is itself 20 AA letters. **Both encoders are in exactly the same position: an
  AA-trained encoder fed a foreign alphabet through an AA vocabulary.** That symmetry is what makes the
  comparison fair. It is *not* a measurement of ESM-2's structural-alphabet ability, and this notebook
  must never be used to claim "SNNEED beats ESM-2 at transfer".

### Powering — read this before quoting any AA number

The AA feed has **5** pairs at `normLev >= 0.70` in the whole 55.1M-pair pool. Therefore:

- **AA Spearman is well powered** (~1,200 stratified pairs) but is dominated by the far band; it measures
  ordering among near-indistinguishable dissimilar pairs.
- **AA AUROC, MAP@10 and RMSE ride on 5 positives / 10 directed queries.** They are anecdotes. The figure
  marks them; the deck must too.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib transformers --quiet

In [ ]:
import json, platform, subprocess, sys, datetime

def _v(mod):
    try: return __import__(mod).__version__
    except Exception as e: return f'<unavailable: {e}>'

ENV = {'captured_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
       'python': sys.version.split()[0], 'platform': platform.platform()}
for m in ['torch', 'numpy', 'pandas', 'scipy', 'sklearn', 'rapidfuzz', 'matplotlib', 'transformers']:
    ENV[m] = _v(m)
try:
    import torch as _t
    ENV['cuda_available'] = _t.cuda.is_available()
    ENV['cuda_device'] = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    ENV['cuda_version'] = _t.version.cuda
except Exception: pass
try: ENV['git_commit'] = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
except Exception: ENV['git_commit'] = '<unknown>'

# METHODS TODO — neither of these is recorded anywhere in the repo. Fill in and keep.
ENV['cath_release'] = 'TODO - exact CATH release, S20 file name, download date'
ENV['3di_source']   = 'TODO - Foldseek version used to generate the 3Di strings'

with open('environment_colab35.json','w') as fh: json.dump(ENV, fh, indent=2)
for k, v in ENV.items(): print(f'{k:<16} {v}')

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# --- identical to colab32 / colab34 so every number is drop-in comparable ---
QUICK   = False                    # True -> SEEDS=[0] smoke pass
N_TRAIN = 30_000
SEEDS   = [0] if QUICK else [0, 1, 2]
EPOCHS  = 30
STRAT_PER_BIN, STRAT_CAND = 400, 200_000
SYN_PERTURB, SYN_INDEP    = 20_000, 8_000
ESM2_MODEL   = 'facebook/esm2_t12_35M_UR50D'
FEED_ORDER   = ['synth', '3Di', 'SS', 'AA']
METHOD_ORDER = ['SNNEED', 'ESM-2', 'Dice']
print(f'seeds={SEEDS}  epochs={EPOCHS}  n_train={N_TRAIN}')

## 2. The final encoder

Unchanged from colab34's `reg-flat` arm. The only difference from the previously-deployed `reg-band`
is that `band_w` is gone: the loss is `mean((pred - normLev)^2)`.

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub','del'])
        else: op = rng.choice(['sub','ins','del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))

In [ ]:
class EncPool(nn.Module):
    # emb -> 2x Conv1d -> AdaptiveAvgPool1d(K) -> fc -> L2 normalise.  THE DELIVERABLE.
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)

class RegModel(nn.Module):
    # parameter-free readout: what is trained is what is deployed
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

class PairDS(Dataset):
    def __init__(s, pp): s.p = pp
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]; return encode_pad(a), encode_pad(b), torch.tensor(l, dtype=torch.float32)

def build_pairs(n, seed):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

def train_snneed(pairs, seed):
    # FINAL OBJECTIVE: plain, unweighted MSE. No head, no bins, no band weights.
    torch.manual_seed(seed); model = RegModel(EncPool()).to(device)
    dl = DataLoader(PairDS(pairs), batch_size=BS, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    model.train(); t0 = time.time()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            loss = ((model(a, b) - y)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'    [SNNEED s={seed}] epoch {ep:>2}/{EPOCHS}  MSE {tot/nb:.5f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); print(f'    [SNNEED s={seed}] trained in {time.time()-t0:.0f}s')
    return model

N_PARAM = sum(p.numel() for p in EncPool().parameters())
print(f'SNNEED encoder parameters: {N_PARAM:,}   (ESM-2 t12 35M: ~35,000,000)')

## 3. Pools, oracles, stratified pairs, audit

Identical construction to colab34, which is the verified one. The audit re-prints so this run carries its
own provenance rather than inheriting a claim.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

# METHODS TODO — outcome-aware filter: these two domains were added after observing that they create
# high-similarity AA pairs. Cannot be stated as a generic rule. See METHODS_OUTLINE.md 3.4.
RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
CATH_FEEDS = ['AA', 'SS', '3Di']
for f in CATH_FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

In [ ]:
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs)
    T_high = {}; pos = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos)

ORACLE = {}
for f in CATH_FEEDS:
    t0 = time.time(); print(f'oracle {f} (SS is the slow one)...')
    ORACLE[f] = build_oracle(f)
    print(f'  {f}: queries@0.70={len(ORACLE[f]["T_high"])}, pos pairs={len(ORACLE[f]["pos_pairs"])}  [{time.time()-t0:.0f}s]')

In [ ]:
def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if ORACLE[feed]['pos_pairs']:
        pa = np.array(ORACLE[feed]['pos_pairs'], float)
        a  = np.concatenate([a, pa[:, 0].astype(np.int64)])
        b  = np.concatenate([b, pa[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, pa[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size:
            t = rng.permutation(idx)[:STRAT_PER_BIN]; ai.append(a[t]); aj.append(b[t]); av.append(nl[t])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64),
                nl=np.concatenate(av))
STRAT = {f: build_strat_pairs(f, np.random.default_rng(999)) for f in CATH_FEEDS}

def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(n_indep): recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]; nl = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL)

SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ; ORACLE['synth'] = build_oracle('synth')
print(f'synth: pool={len(SYN_SEQ)}, queries@0.70={len(ORACLE["synth"]["T_high"])}')

In [ ]:
def _pairs(feed):
    if feed == 'synth': return SYN_I, SYN_J, SYN_NL
    P = STRAT[feed]; return P['i'], P['j'], P['nl']

print(f'{"feed":<7}{"pool":>7}{"queries@.70":>13}{"oracle pos":>12}'
      f'{"strat n":>10}{"strat>=.70":>12}{"strat<.30":>11}{"strat mid":>11}')
print('-'*83)
AUDIT = {}
for f in FEED_ORDER:
    I, J, nl = _pairs(f)
    row = dict(pool=len(POOL_SEQ[f]), queries_at_070=len(ORACLE[f]['T_high']),
               oracle_pos_pairs=len(ORACLE[f]['pos_pairs']), strat_n=int(len(nl)),
               strat_high=int((nl >= BAND_HIGH).sum()), strat_far=int((nl < BAND_LOW_AA).sum()),
               strat_mid=int(((nl >= BAND_LOW_AA) & (nl < BAND_HIGH)).sum()),
               strat_median=float(np.median(nl)))
    AUDIT[f] = row
    print(f'{f:<7}{row["pool"]:>7}{row["queries_at_070"]:>13}{row["oracle_pos_pairs"]:>12}'
          f'{row["strat_n"]:>10}{row["strat_high"]:>12}{row["strat_far"]:>11}{row["strat_mid"]:>11}')

print('\nEXPECTED (colab34, verified):')
print('  synth 7296/2410 | 3Di 10501/347 | SS 10497/10002 | AA 10501/10')
print('  If AA queries@.70 != 10 or 3Di != 347, the pool changed — stop and diff before using the numbers.')
for f in FEED_ORDER:
    if AUDIT[f]['queries_at_070'] == 0:
        print(f'  *** WARNING: {f} has ZERO queries at >=0.70 — AUROC/MAP@10 will be NaN (this is the colab33 failure mode).')
    elif AUDIT[f]['queries_at_070'] <= 20:
        print(f'  *** CAUTION: {f} has only {AUDIT[f]["queries_at_070"]} queries at >=0.70 — '
              f'AUROC/MAP@10/RMSE for {f} are anecdotes. Spearman is still well powered '
              f'(n={AUDIT[f]["strat_n"]}) but is dominated by the far band.')
with open('colab35_audit.json','w') as fh: json.dump(AUDIT, fh, indent=2)

## 4. Metrics and the three scorers

All three methods are scored **identically**: a similarity per stratified pair, plus a full-pool top-10
retrieval against the exhaustive Levenshtein oracle. Spearman is additionally decomposed by band.

**RMSE is reported for SNNEED only.** SNNEED's readout is trained to *be* `normLev`, so absolute error is
meaningful. ESM-2 cosine and Dice overlap are similarity scores on their own arbitrary scales — an RMSE
against `normLev` would measure their calibration, not their quality, and would be an unfair comparison.

In [ ]:
def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan

def _rho(sim, nl):
    if len(nl) < 10 or np.ptp(nl) == 0: return np.nan
    r = spearmanr(sim, nl).correlation
    return float(r) if r == r else np.nan

BANDS = {'far': lambda nl: nl < BAND_LOW_AA,
         'mid': lambda nl: (nl >= BAND_LOW_AA) & (nl < BAND_HIGH),
         'high': lambda nl: nl >= BAND_HIGH}

def map10_emb(E_t, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]; sc = E_t[qi] @ E_t.t()
        for r, idx in enumerate(qi): sc[r, idx] = -1e9
        top = torch.topk(sc, k, dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi):
            ts = set(T_high[idx].tolist()); hits = 0; ap = 0.0
            for rr, o in enumerate(top[r], 1):
                if o in ts: hits += 1; ap += hits / rr
            aps.append(ap / min(len(ts), k))
    return float(np.mean(aps))

def _record(method, feed, sim, nl, map10, rmse_high=np.nan, seed=0):
    rec = dict(method=method, feed=feed, seed=seed, spearman=_rho(sim, nl),
               auroc=_auroc(sim, nl), map10=map10, rmse_high=rmse_high,
               n_pairs=int(len(nl)), n_queries=len(ORACLE[feed]['T_high']))
    for bn, bf in BANDS.items():
        m = bf(nl); rec[f'spearman_{bn}'] = _rho(sim[m], nl[m]); rec[f'n_{bn}'] = int(m.sum())
    return rec

@torch.no_grad()
def snn_embed(model, feed, bs=256):
    seqs = POOL_SEQ[feed]; out = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

def eval_snneed(model, feed, seed):
    E = snn_embed(model, feed); I, J, nl = _pairs(feed)
    sim = np.sum(E[I] * E[J], axis=1)                       # cosine, as deployed
    pred = 1.0 - np.linalg.norm(E[I] - E[J], axis=1) / 2.0  # native normLev-scale readout
    hm = nl >= BAND_HIGH
    rmse = float(np.sqrt(np.mean((pred[hm] - nl[hm])**2))) if hm.sum() else np.nan
    return _record('SNNEED', feed, sim, nl,
                   map10_emb(torch.as_tensor(E, device=device), ORACLE[feed]['T_high']),
                   rmse, seed)

In [ ]:
# --- Dice: binary 3-gram set overlap, 2|A^B| / (|A|+|B|) ---
def build_kmer(feed, k=3):
    seqs = POOL_SEQ[feed]; vocab = {}; rows = []; cols = []
    for i, s in enumerate(seqs):
        for g in set(s[t:t+k] for t in range(len(s)-k+1)):
            rows.append(i); cols.append(vocab.setdefault(g, len(vocab)))
    B = sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(seqs), max(1, len(vocab))))
    return B, np.asarray(B.sum(1)).ravel(), len(vocab)

def eval_dice(feed):
    B, sz, nvocab = build_kmer(feed); I, J, nl = _pairs(feed)
    inter = np.asarray(B[I].multiply(B[J]).sum(1)).ravel().astype(float)
    sim = 2 * inter / np.maximum(sz[I] + sz[J], 1e-9)
    T = ORACLE[feed]['T_high']; aps = []
    for qi in T:
        it = np.asarray(B.dot(B[qi].T).todense()).ravel().astype(float)
        s = 2 * it / np.maximum(sz + sz[qi], 1e-9); s[qi] = -1.0
        order = np.argpartition(-s, min(10, len(s)-1))[:10]; order = order[np.argsort(-s[order])]
        ts = set(T[qi].tolist()); hits = 0; ap = 0.0
        for rr, o in enumerate(order, 1):
            if o in ts: hits += 1; ap += hits / rr
        aps.append(ap / min(len(ts), 10))
    print(f'  Dice {feed:<6} distinct 3-grams observed = {nvocab:>6}  '
          f'(alphabet^3 ceiling: SS 27, 20-letter 8000)')
    return _record('Dice', feed, sim, nl, float(np.mean(aps)) if aps else np.nan)

In [ ]:
# --- ESM-2: frozen, mean-pooled over real residues (BOS and EOS masked out) ---
_esm = {}
@torch.no_grad()
def esm_embed(feed, bs=32):
    cf = f'colab35_esm2_{feed}.npy'
    if os.path.exists(cf): return np.load(cf)
    if 'mdl' not in _esm:
        from transformers import AutoTokenizer, AutoModel
        _esm['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
        _esm['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
    tok, mdl = _esm['tok'], _esm['mdl']; seqs = POOL_SEQ[feed]
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]; batch = [seqs[j] for j in idx]
        enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=True).to(device)
        h = mdl(**enc).last_hidden_state; mask = enc['attention_mask'].clone(); mask[:, 0] = 0
        for r, l in enumerate(enc['attention_mask'].sum(1)): mask[r, l-1] = 0
        m = mask.unsqueeze(-1).float()
        e = F.normalize((h*m).sum(1)/m.sum(1).clamp(min=1), dim=1).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    E = np.stack(out).astype(np.float32); np.save(cf, E); return E

def eval_esm(feed):
    E = esm_embed(feed); I, J, nl = _pairs(feed)
    sim = np.sum(E[I] * E[J], axis=1)
    return _record('ESM-2', feed, sim, nl,
                   map10_emb(torch.as_tensor(E, device=device), ORACLE[feed]['T_high']))

## 5. Run

In [ ]:
rows = []
for seed in SEEDS:
    print(f'\n=========== SNNEED seed {seed} ===========')
    pairs = build_pairs(N_TRAIN, seed)
    lab = np.array([l for *_, l in pairs])
    print(f'  training pairs: n={len(pairs)}  median normLev={np.median(lab):.3f}  '
          f'far={int((lab<BAND_LOW_AA).sum())} mid={int(((lab>=BAND_LOW_AA)&(lab<BAND_HIGH)).sum())} '
          f'high={int((lab>=BAND_HIGH).sum())}')
    print('  ^ note the far count: the 20-letter chance floor (~0.28) makes the far band nearly empty.')
    model = train_snneed(pairs, seed)
    for feed in FEED_ORDER: rows.append(eval_snneed(model, feed, seed))
    got = {r['feed']: r['spearman'] for r in rows[-len(FEED_ORDER):]}
    print('    -> ' + '  '.join(f'{f}:rho={got[f]:+.2f}' for f in FEED_ORDER))
    if seed == SEEDS[-1]:
        torch.save(model.encoder.state_dict(), 'colab35_snneed_encoder.pt')
        print('    saved colab35_snneed_encoder.pt (last seed) — the deployable artefact')
    del model
    if device.type == 'cuda': torch.cuda.empty_cache()

In [ ]:
print('ESM-2 (frozen — deterministic, one pass):')
for feed in FEED_ORDER:
    r = eval_esm(feed); rows.append(r)
    print(f'  ESM-2 {feed:<6} rho={r["spearman"]:+.3f}  AUROC={r["auroc"]:.3f}  MAP@10={r["map10"]:.3f}')

In [ ]:
print('Dice (classical, deterministic):')
for feed in FEED_ORDER:
    r = eval_dice(feed); rows.append(r)
    print(f'  Dice  {feed:<6} rho={r["spearman"]:+.3f}  AUROC={r["auroc"]:.3f}  MAP@10={r["map10"]:.3f}')

df = pd.DataFrame(rows); df.to_csv('colab35_metrics.csv', index=False)
print(f'\nsaved colab35_metrics.csv ({len(df)} rows)')

## 6. Results

In [ ]:
agg = (df.groupby(['method','feed'])
         .agg(spearman=('spearman','mean'), sp_sd=('spearman','std'),
              sp_far=('spearman_far','mean'), sp_mid=('spearman_mid','mean'),
              sp_high=('spearman_high','mean'), auroc=('auroc','mean'),
              map10=('map10','mean'), rmse_high=('rmse_high','mean'),
              n_pairs=('n_pairs','first'), n_high=('n_high','first'),
              n_queries=('n_queries','first')).reset_index())
agg['method'] = pd.Categorical(agg['method'], METHOD_ORDER, ordered=True)
agg['feed']   = pd.Categorical(agg['feed'], FEED_ORDER, ordered=True)
agg = agg.sort_values(['feed','method'])
pd.set_option('display.width', 220, 'display.max_columns', 40)
print(agg.to_string(index=False, float_format=lambda x: f'{x:7.3f}'))
agg.to_csv('colab35_summary.csv', index=False)

In [ ]:
for m, name in [('spearman','Spearman rho (rank fidelity)'),
                ('auroc','AUROC (high vs background)'),
                ('map10','MAP@10 (full-pool retrieval)')]:
    print(f'\n=== {name} ===')
    print(agg.pivot(index='method', columns='feed', values=m)
             .to_string(float_format=lambda x: f'{x:6.3f}'))

print('\n=== Powering — quote these with every AA number ===')
for f in FEED_ORDER:
    a = agg[agg.feed == f].iloc[0]
    print(f'  {f:<6} Spearman on n={int(a.n_pairs):>5} pairs | AUROC/MAP on '
          f'{int(a.n_high):>5} positives / {int(a.n_queries):>5} queries')

print('\n=== SNNEED value fidelity (RMSE vs normLev on >=0.70; SNNEED only, by design) ===')
print(agg[agg.method == 'SNNEED'][['feed','rmse_high','n_high']]
        .to_string(index=False, float_format=lambda x: f'{x:6.3f}'))

print('\n=== Seed stability of SNNEED (sd over seeds) ===')
print(agg[agg.method == 'SNNEED'][['feed','spearman','sp_sd']]
        .to_string(index=False, float_format=lambda x: f'{x:6.3f}'))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
FEED_C = {'synth':'#E8871A', '3Di':'#2E6DB4', 'SS':'#C0392B', 'AA':'#8A8F98'}
cmap = LinearSegmentedColormap.from_list('deck', ['#FFFFFF', '#F3C9C0', '#C0392B', '#7B1E12'])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.4))
for ax, (metric, title, vmin) in zip(axes, [('spearman','SPEARMAN rho — captures ranking?', -1),
                                            ('auroc','AUROC — separates high from background?', 0),
                                            ('map10','MAP@10 — retrieves true neighbours?', 0)]):
    M = agg.pivot(index='method', columns='feed', values=metric).reindex(index=METHOD_ORDER, columns=FEED_ORDER)
    im = ax.imshow(M.values, cmap=cmap, vmin=vmin, vmax=1.0, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M.values[i, j]
            ax.text(j, i, 'n/a' if v != v else f'{v:.2f}', ha='center', va='center',
                    color='white' if (v == v and v > 0.72) else '#222', fontsize=11,
                    fontweight='bold' if M.index[i] == 'SNNEED' else 'normal')
    ax.set_xticks(range(len(FEED_ORDER))); ax.set_xticklabels(FEED_ORDER, fontsize=11)
    for t, f in zip(ax.get_xticklabels(), FEED_ORDER): t.set_color(FEED_C[f])
    ax.set_yticks(range(len(METHOD_ORDER))); ax.set_yticklabels(METHOD_ORDER, fontsize=11)
    ax.set_title(title, fontsize=11)
    if metric in ('auroc', 'map10'):     # mark the underpowered column
        ax.add_patch(plt.Rectangle((len(FEED_ORDER)-1.5, -0.5), 1, len(METHOD_ORDER),
                                   fill=False, ec='#333', lw=2, ls='--'))
        ax.set_xlabel('dashed: AA has only 5 positives / 10 queries', fontsize=8.5)
    plt.colorbar(im, ax=ax, fraction=0.035)
plt.suptitle('colab35 — SNNEED (reg+pool, plain MSE) vs ESM-2 vs Dice   '
             f'[SNNEED mean of seeds {SEEDS}; ESM-2 and Dice deterministic]', fontsize=12)
plt.tight_layout(); plt.savefig('colab35_heatmaps.png', dpi=150, bbox_inches='tight'); plt.show()

### 6a. Cost — the missing "better than classical" slide

The argument is asymptotic, not just a stopwatch reading:

- **Exact Levenshtein:** `O(n*m)` **per pair**, recomputed for every comparison. A full-pool query costs
  `N` dynamic-programming runs.
- **SNNEED:** `O(n)` **once per sequence** to embed, then `O(d)` per comparison — and because the output
  is a plain 128-d vector, the pool is **indexable**, so an ANN index makes retrieval sub-linear.
- **ESM-2:** also embed-once, but the forward pass is ~100x the parameters.

**Do not quote a single speed-up number.** `rapidfuzz` is optimised C running on all cores, and at small
pool sizes the encode cost dominates, so SNNEED is genuinely *slower*. The cell below therefore reports
the **crossover pool size** and a projection across N. The defensible slide claim is asymptotic and about
indexability, not a stopwatch reading. State the hardware whenever any of it is quoted.

In [ ]:
BENCH_FEED = 'AA'; seqs = POOL_SEQ[BENCH_FEED]; NB_ = min(512, len(seqs)); sub = seqs[:NB_]
bench = {}

m = RegModel(EncPool()).to(device).eval()
with torch.no_grad():                                    # warm up
    m.encoder(torch.stack([encode_pad(s) for s in sub[:32]]).to(device))
if device.type == 'cuda': torch.cuda.synchronize()
t0 = time.time()
with torch.no_grad():
    for i in range(0, NB_, 256):
        m.encoder(torch.stack([encode_pad(s) for s in sub[i:i+256]]).to(device))
if device.type == 'cuda': torch.cuda.synchronize()
bench['snneed_embed_s_per_seq'] = (time.time() - t0) / NB_
del m

if os.path.exists(f'colab35_esm2_{BENCH_FEED}.npy'):
    from transformers import AutoTokenizer, AutoModel
    if 'mdl' not in _esm:
        _esm['tok'] = AutoTokenizer.from_pretrained(ESM2_MODEL)
        _esm['mdl'] = AutoModel.from_pretrained(ESM2_MODEL).to(device).eval()
    tok, mdl = _esm['tok'], _esm['mdl']
    with torch.no_grad():
        mdl(**tok(sub[:8], return_tensors='pt', padding=True).to(device))
    if device.type == 'cuda': torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for i in range(0, NB_, 32):
            mdl(**tok(sub[i:i+32], return_tensors='pt', padding=True).to(device))
    if device.type == 'cuda': torch.cuda.synchronize()
    bench['esm2_embed_s_per_seq'] = (time.time() - t0) / NB_

t0 = time.time()
rf_cdist(sub[:64], seqs, scorer=RFLev.distance, workers=-1)
bench['exact_lev_s_per_pair'] = (time.time() - t0) / (64 * len(seqs))

N = len(seqs)
bench['pool_size'] = N
bench['exact_query_s']  = bench['exact_lev_s_per_pair'] * N
bench['snneed_query_s'] = bench['snneed_embed_s_per_seq'] + N * 128 * 2e-10
bench['speedup_vs_exact'] = bench['exact_query_s'] / bench['snneed_query_s']
bench['snneed_params'] = int(N_PARAM)
bench['device'] = str(device)
bench['gpu'] = torch.cuda.get_device_name(0) if device.type == 'cuda' else platform.processor()

print(f'pool = {N} sequences ({BENCH_FEED}), device = {bench["gpu"]}\n')
print(f'  SNNEED embed          {bench["snneed_embed_s_per_seq"]*1e3:8.3f} ms / sequence   '
      f'({bench["snneed_params"]:,} params)')
if 'esm2_embed_s_per_seq' in bench:
    print(f'  ESM-2  embed          {bench["esm2_embed_s_per_seq"]*1e3:8.3f} ms / sequence   (~35M params)'
          f'   -> SNNEED is {bench["esm2_embed_s_per_seq"]/bench["snneed_embed_s_per_seq"]:.0f}x faster to embed')
print(f'  exact Levenshtein     {bench["exact_lev_s_per_pair"]*1e6:8.3f} us / pair (rapidfuzz, C, all cores)')
print(f'\n  ONE full-pool query against {N} sequences:')
print(f'    exact DP   {bench["exact_query_s"]*1e3:9.1f} ms   (N Levenshtein runs, every query)')
print(f'    SNNEED     {bench["snneed_query_s"]*1e3:9.1f} ms   (1 encode + N dot products; pool embedded once)')
print(f'    speed-up   {bench["speedup_vs_exact"]:9.1f}x')

# The single-number speed-up is pool-size dependent and must NOT be quoted alone.
# rapidfuzz is heavily optimised C on all cores; at small N the encode dominates and SNNEED LOSES.
bench['crossover_pool_n'] = bench['snneed_embed_s_per_seq'] / bench['exact_lev_s_per_pair']
print(f'\n  Projected cost of ONE query, pool embedded once and amortised:')
print(f'    {"pool N":>12} {"exact DP":>12} {"SNNEED":>12} {"speed-up":>10}')
proj = {}
for Np in [1e4, 1e5, 1e6, 1e7]:
    ex = bench['exact_lev_s_per_pair'] * Np
    sn = bench['snneed_embed_s_per_seq'] + Np * 128 * 2e-10
    proj[int(Np)] = dict(exact_s=ex, snneed_s=sn, speedup=ex/sn)
    print(f'    {int(Np):>12,} {ex*1e3:>10.1f}ms {sn*1e3:>10.1f}ms {ex/sn:>9.1f}x')
bench['projection'] = proj
print(f'\n  CROSSOVER: SNNEED only wins once the pool exceeds ~{bench["crossover_pool_n"]:,.0f} sequences.')
print('  Below that, exact DP is cheaper — say so on the slide. The claim is ASYMPTOTIC and it is')
print('  about indexability: the pool is embedded ONCE, and a 128-d vector index (ANN) makes retrieval')
print('  sub-linear in N. Exact DP is Theta(N) per query forever, and cannot be indexed.')
with open('colab35_bench.json','w') as fh: json.dump(bench, fh, indent=2)

In [ ]:
outs = ['colab35_metrics.csv','colab35_summary.csv','colab35_audit.json','colab35_bench.json',
        'colab35_heatmaps.png','colab35_snneed_encoder.pt','environment_colab35.json']
for f in outs:
    print(f'  {f}  ({os.path.getsize(f):,} bytes)' if os.path.exists(f) else f'  MISSING {f}')
try:
    from google.colab import files
    for f in outs:
        if os.path.exists(f): files.download(f)
except Exception as e:
    print('not on Colab / download skipped:', e)

## 7. What goes on which slide

| Output | Slide | Note |
|---|---|---|
| `colab35_heatmaps.png` | 21, 26, 27 (replaces all three) | AA column is boxed for AUROC/MAP — keep that marking |
| Powering block | 21, 26, 27 | say `n = 5` out loud before anyone asks |
| Cost block | **new speed/scaling slide (S6)** | quote the hardware line with it |
| `colab35_snneed_encoder.pt` | — | the deliverable artefact; 128-d vectors, indexable |
| `environment_colab35.json` | Methods | library versions; still needs `cath_release` and `3di_source` filled in by hand |

### Reading rules — carry these into the prose

1. **Dice on SS is the demonstration, not an embarrassment.** SS has 3 letters, so there are only
   `3^3 = 27` possible 3-grams and nearly every sequence contains nearly all of them. The `distinct
   3-grams observed` line printed by `eval_dice` shows this directly. That is *why* Dice collapses on SS
   and why it is near-perfect on synth (8,000 possible 3-grams, near-unique). It is an argument about what
   a learned metric buys, not a defeat.
2. **Never write "SNNEED beats ESM-2 at transfer."** On SS/3Di, ESM-2 is a control: both encoders read
   `H` as histidine. The defensible sentence is *"the transfer SNNEED achieves is not something you get
   for free from any large pretrained encoder fed the same characters."*
3. **On AA, report Spearman and MAP@10 as different kinds of evidence.** Spearman there is well powered
   but measures ordering among near-indistinguishable dissimilar pairs; MAP@10 rides on 10 queries.
   Whichever way they point, say which is which.
4. **If Dice wins a column, say so.** It wins synth by construction and may win AA. Conceding a
   well-understood loss is what makes the SS/3Di result credible.